# Río Mana — SegmentEveryGrain, procesamiento completo por teselas

Versión **experimental para Colab**. Basada en el cuaderno de Zoltán Sylvester y el piloto del río Mana. JPEG de ~1 mm/píxel, worldfile y **EPSG:32619**. Conserva la resolución original. Produce resultados por tesela y un GeoPackage final. **Primero ejecutar `MAX_TESELAS=2` y comprobar las salidas.**

Los objetos se asignan a una sola tesela por su centroide en la zona interior (core). Esto evita duplicados habituales, pero los clastos cortados en los bordes de las ventanas **no se reconstruyen**: se marcan como posibles truncamientos. La segmentación y las medidas requieren validación. Las salidas intermedias se guardan en Drive para reanudar.

## 1. Instalación (GPU T4 recomendada)

In [ ]:
import sys
!{sys.executable} -m pip -q install segmenteverygrain sam2 rtree geopandas pyogrio affine
!wget -q -nc https://huggingface.co/facebook/sam2.1-hiera-large/resolve/main/sam2.1_hiera_large.pt
!wget -q -nc https://raw.githubusercontent.com/zsylvester/segmenteverygrain/main/models/seg_model_smooth_labels.pt


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Modelos preentrenados

In [ ]:
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
import segmenteverygrain as seg
import segmenteverygrain.interactions as si
from sam2.build_sam import build_sam2
from pathlib import Path
from PIL import Image
from affine import Affine
from shapely.geometry import shape
from shapely.ops import transform as shapely_transform
import geopandas as gpd
import json, gc, time, traceback

assert torch.cuda.is_available(), 'Activar GPU en Colab antes de continuar'
sam = build_sam2('configs/sam2.1/sam2.1_hiera_l.yaml',
                 'sam2.1_hiera_large.pt', device='cuda')
unet = seg.load_unet('seg_model_smooth_labels.pt')
print('GPU:', torch.cuda.get_device_name(0))

GPU: Tesla T4


## 3. Configuración del ortomosaico y salida

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CARPETA = Path('/content/drive/MyDrive/gravas_mana')  # AJUSTAR
JPG = CARPETA / 'odm_orthophoto_jpg.jpg'               # AJUSTAR
WLD = CARPETA / 'odm_orthophoto_jpg.wld'           # AJUSTAR
SALIDA = CARPETA / 'resultados_completo'
SALIDA.mkdir(parents=True, exist_ok=True)
EPSG = 'EPSG:32619'
TAM = 1024
SOLAPE = 128
MIN_AREA = 100.0       # Cambiar a 400.0 si se desea replicar piloto original
DBS_MAX_DIST = 20.0
MAX_TESELAS = None        # Número de teselas CON IMAGEN VÁLIDA por sesión; None = todas
MODO_PRUEBA = False    # True: comenzar cerca de (10000, 8000); False: orden normal
PUNTO_PRUEBA = (10000, 8000)
UMBRAL_SIN_DATOS = 0.98  # fracción de píxeles negros para descartar tesela
VALOR_NEGRO = 5          # RGB < 5 en todos los canales
REANUDAR = True
GUARDAR_PNG = True
assert JPG.exists() and WLD.exists(), 'Revisar JPG y WLD'
A,D,B,E,C,F = map(float, WLD.read_text().split())
transform = Affine(A,B,C-(A+B)/2,D,E,F-(D+E)/2)
px_per_m = 1 / ((abs(A)+abs(E))/2)
Image.MAX_IMAGE_PIXELS = None
with Image.open(JPG) as im:
    ANCHO, ALTO = im.size
print('Dimensiones:',ANCHO,ALTO,'px; escala:',px_per_m,'px/m')
print('CRS:',EPSG, 'Transformación:', transform)

Mounted at /content/drive
Dimensiones: 24571 19904 px; escala: 1000.0280507868246 px/m
CRS: EPSG:32619 Transformación: | 0.00, 0.00, 371854.73|
| 0.00,-0.00, 2057666.48|
| 0.00, 0.00, 1.00|


## 4. Retícula con solape y zona de responsabilidad única

Las teselas tienen paso `TAM - SOLAPE`. La zona interior asignada a cada tesela se separa por la mitad del solape con la tesela vecina. Se conservan solo los granos cuyo centroide cae en esa zona. Los granos que tocan el borde físico de su ventana se señalan en la tabla.

In [ ]:
def inicios(n, tam, paso):
    valores = list(range(0, n, paso))
    return [v for v in valores if v < n]

PASO = TAM - SOLAPE
xs, ys = inicios(ANCHO,TAM,PASO), inicios(ALTO,TAM,PASO)

def ventanas():
    for iy,y in enumerate(ys):
        for ix,x in enumerate(xs):
            w,h = min(TAM,ANCHO-x), min(TAM,ALTO-y)
            izq = 0 if ix == 0 else SOLAPE/2
            sup = 0 if iy == 0 else SOLAPE/2
            der = w if ix == len(xs)-1 else min(w, PASO+SOLAPE/2)
            inf = h if iy == len(ys)-1 else min(h, PASO+SOLAPE/2)
            yield dict(id=f'x{x:05d}_y{y:05d}', x=x,y=y,w=w,h=h,
                       core=(izq,sup,der,inf))

plan = list(ventanas())
print('Teselas previstas:',len(plan),'(',len(xs),'columnas ×',len(ys),'filas )')
print('Primeras:',plan[:2])

Teselas previstas: 644 ( 28 columnas × 23 filas )
Primeras: [{'id': 'x00000_y00000', 'x': 0, 'y': 0, 'w': 1024, 'h': 1024, 'core': (0, 0, 960.0, 960.0)}, {'id': 'x00896_y00000', 'x': 896, 'y': 0, 'w': 1024, 'h': 1024, 'core': (64.0, 0, 960.0, 960.0)}]


## 5. Procesar teselas, exportar máscaras vistosas y mediciones

Se guardan por tesela: PNG de segmentación, CSV de medidas, GeoJSON en UTM y un archivo `ok` al terminar. La exportación de figuras utiliza el `fig` devuelto por `sam_segmentation`, no el editor interactivo. Se emplean las funciones originales de SegmentEveryGrain para medir y resumir.

In [ ]:
def procesar_tesela(v):
    tid=v['id']; carpeta=SALIDA/tid
    carpeta.mkdir(exist_ok=True)
    ok=carpeta/'completado.ok'
    if REANUDAR and ok.exists():
        print('OMITIDA (completada):',tid)
        return 'omitida'
    inicio=time.perf_counter()
    with Image.open(JPG) as im:
        image=np.asarray(im.crop((v['x'],v['y'],v['x']+v['w'],v['y']+v['h'])).convert('RGB'))
    # Descartar zonas negras antes de invocar U-Net y SAM.
    fraccion_negra = float(np.mean(np.all(image < VALOR_NEGRO, axis=2)))
    if fraccion_negra >= UMBRAL_SIN_DATOS:
        duracion = time.perf_counter() - inicio
        (carpeta/'registro.json').write_text(json.dumps({
            'tesela': tid, 'estado': 'sin_datos', 'fraccion_negra': fraccion_negra,
            'semillas': 0, 'mascaras': 0, 'granos_conservados': 0,
            'segundos': duracion, 'min_area': MIN_AREA,
            'dbs_max_dist': DBS_MAX_DIST, 'toca_borde': 0
        }, indent=2))
        ok.write_text('SIN_DATOS')
        print(tid, 'sin datos:', f'{fraccion_negra:.1%} negro; omitida')
        return 'sin_datos'
    image_pred=seg.predict_image(image,unet,I=256)
    labels,coords=seg.label_grains(image,image_pred,dbs_max_dist=DBS_MAX_DIST)
    # No ejecutar SAM cuando no hay semillas.
    if len(coords) == 0:
        duracion = time.perf_counter() - inicio

        registro = {
            "tesela": tid,
            "estado": "sin_semillas",
            "fraccion_negra": fraccion_negra,
            "semillas": 0,
            "mascaras": 0,
            "granos_conservados": 0,
            "segundos": duracion,
            "min_area": MIN_AREA,
            "dbs_max_dist": DBS_MAX_DIST,
            "toca_borde": 0
        }

        (carpeta / "registro.json").write_text(
            json.dumps(registro, indent=2)
        )

        # Guardar una imagen para revisar posteriormente
        # las teselas sin detecciones.
        Image.fromarray(image).save(
            carpeta / "tesela_sin_semillas.png"
        )

        # Marcar como procesada, sin confundirla con una
        # tesela que contiene clastos.
        (carpeta / "completado.ok").write_text(
            "SIN_SEMILLAS"
        )

        print(
            tid,
            "sin semillas; se continúa con la siguiente tesela"
        )

        return "sin_semillas"
    all_grains,labels,mask_all,grain_data,fig,ax=seg.sam_segmentation(
        sam,image,image_pred,coords,labels,min_area=MIN_AREA,
        plot_image=True,remove_edge_grains=False,remove_large_objects=False)
    if GUARDAR_PNG and fig is not None:
        fig.savefig(carpeta/'segmentacion.png',dpi=180,bbox_inches='tight')
    if fig is not None: plt.close(fig)
    grains=si.polygons_to_grains(all_grains,image=image)
    for g in grains: g.measure()
    # Exportaciones nativas: conservar medidas de SEG sin redefinir ejes.
    si.save_grains(str(carpeta/'granos_local.geojson'),grains)
    resumen=si.save_summary(str(carpeta/'resumen_original.csv'),grains,px_per_m=px_per_m)
    if not isinstance(resumen,pd.DataFrame):
        resumen=pd.read_csv(carpeta/'resumen_original.csv')
    geo=json.loads((carpeta/'granos_local.geojson').read_text())
    feats=geo.get('features',[])
    if len(feats)!=len(resumen):
        raise ValueError(f'{tid}: {len(feats)} geometrías y {len(resumen)} filas: revisar correspondencia antes de unir')
    registros=[]; geometrias=[]
    l,t,r,b=v['core']
    for i,feature in enumerate(feats):
        if not feature.get('geometry'): continue
        p=shape(feature['geometry'])
        cx,cy=p.centroid.x,p.centroid.y
        if not (l<=cx<r and t<=cy<b): continue
        # Las coordenadas de las geometrías originales son píxeles locales.
        geom=shapely_transform(lambda xx,yy,z=None: transform*(xx+v['x'],yy+v['y']),p)
        fila=resumen.iloc[i].to_dict()
        fila.update(id_grano=f'{tid}_g{i:05d}',tesela=tid,
                    pixel_col=v['x']+cx,pixel_fila=v['y']+cy,
                    x_utm=geom.centroid.x,y_utm=geom.centroid.y,
                    toca_borde=(p.bounds[0]<=1 or p.bounds[1]<=1 or
                                p.bounds[2]>=v['w']-1 or p.bounds[3]>=v['h']-1))
        registros.append(fila); geometrias.append(geom)
    gdf=gpd.GeoDataFrame(registros,geometry=geometrias,crs=EPSG)
    # GeoJSON por tesela para poder consolidar sin volver a segmentar.
    gdf.to_file(carpeta/'granos_utm.geojson',driver='GeoJSON')
    gdf.drop(columns='geometry').to_csv(carpeta/'mediciones.csv',index=False)
    duracion=time.perf_counter()-inicio
    (carpeta/'registro.json').write_text(json.dumps({
        'tesela':tid,'estado':'completada','fraccion_negra':fraccion_negra,
        'semillas':len(coords),'mascaras':len(feats),
        'granos_conservados':len(gdf),'segundos':duracion,
        'min_area':MIN_AREA,'dbs_max_dist':DBS_MAX_DIST,
        'toca_borde':int(gdf.toca_borde.sum()) if len(gdf) else 0},indent=2))
    ok.write_text('OK')
    print(tid,'semillas',len(coords),'máscaras',len(feats),
          'conservados',len(gdf),'tiempo',round(duracion,1),'s')
    del image,image_pred,labels,mask_all,grain_data,grains,all_grains,gdf
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return 'completada'

# MAX_TESELAS limita las teselas válidas, no las zonas negras.
pendientes = [v for v in plan if not (
    REANUDAR and (SALIDA/v['id']/'completado.ok').exists()
)]
if MODO_PRUEBA:
    px, py = PUNTO_PRUEBA
    pendientes.sort(key=lambda v: (
        (v['x'] + v['w']/2 - px)**2 +
        (v['y'] + v['h']/2 - py)**2
    ))

print('Pendientes:', len(pendientes))
print('Modo prueba:', MODO_PRUEBA,
      '| límite de teselas con imagen válida:', MAX_TESELAS)
validas = 0
sin_datos = 0
sin_semillas = 0
completadas = 0
for v in pendientes:
    if MAX_TESELAS is not None and validas >= MAX_TESELAS:
        break
    try:
        estado = procesar_tesela(v)
        if estado == 'sin_datos':
            sin_datos += 1
            continue
        validas += 1
        if estado == 'sin_semillas':
            sin_semillas += 1
        elif estado == 'completada':
            completadas += 1
    except Exception as e:
        print('ERROR en', v['id'], repr(e))
        with (SALIDA/'errores.log').open('a') as f:
            f.write(v['id'] + ' ' + repr(e) + '\n')
        traceback.print_exc()
        raise
print('Resumen sesión: válidas:', validas,
      '| segmentadas:', completadas,
      '| sin semillas:', sin_semillas,
      '| sin datos:', sin_datos)


Pendientes: 66
Modo prueba: False | límite de teselas con imagen válida: None


segmenting image tiles: 100%|██████████| 4/4 [00:02<00:00,  1.98it/s]
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/transforms/functional.py:154: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  img = torch.from_numpy(pic.transpose((2, 0, 1))).contiguous()


creating masks using SAM...


100%|██████████| 707/707 [00:28<00:00, 24.48it/s]


finding overlapping polygons...


697it [00:01, 349.30it/s]


... (121 outputs left out)

100%|██████████| 1/1 [00:00<00:00, 54.52it/s]


creating labeled image...


100%|██████████| 2/2 [00:00<00:00, 79.18it/s]


x17024_y19712 semillas 8 máscaras 2 conservados 0 tiempo 6.5 s
x17920_y19712 sin datos: 100.0% negro; omitida
x18816_y19712 sin datos: 100.0% negro; omitida
x19712_y19712 sin datos: 100.0% negro; omitida
x20608_y19712 sin datos: 100.0% negro; omitida
x21504_y19712 sin datos: 100.0% negro; omitida
x22400_y19712 sin datos: 100.0% negro; omitida
x23296_y19712 sin datos: 100.0% negro; omitida
x24192_y19712 sin datos: 100.0% negro; omitida
Resumen sesión: válidas: 13 | segmentadas: 13 | sin semillas: 0 | sin datos: 53


In [ ]:
from pathlib import Path
from collections import Counter
import json
import pandas as pd

# SALIDA debe apuntar al mismo directorio utilizado
# durante la segmentación.
SALIDA = Path(SALIDA)

carpetas = sorted(
    p for p in SALIDA.iterdir()
    if p.is_dir() and p.name.startswith("x")
)

registros = []

for carpeta in carpetas:
    archivo = carpeta / "registro.json"

    if archivo.exists():
        try:
            datos = json.loads(archivo.read_text())
        except (OSError, ValueError):
            datos = {}
    else:
        datos = {}

    completo = (carpeta / "completado.ok").exists()

    registros.append({
        "tesela": carpeta.name,
        "completada": completo,
        "registro": archivo.exists(),
        "csv": (carpeta / "mediciones.csv").exists(),
        "geojson": (carpeta / "granos_utm.geojson").exists(),
        "png": (carpeta / "segmentacion.png").exists(),
        "granos": datos.get("granos_conservados", None),
        "semillas": datos.get("semillas", None)
    })

auditoria = pd.DataFrame(registros)

print("Carpetas encontradas:", len(auditoria))

if not auditoria.empty:
    print("\nArchivos encontrados:")
    print(
        auditoria[
            ["completada", "registro", "csv", "geojson", "png"]
        ].sum()
    )

    print("\nClastos conservados según registros:")
    print(auditoria["granos"].sum())

    incompletas = auditoria[
        ~auditoria[
            ["completada", "registro", "csv", "geojson"]
        ].all(axis=1)
    ]

    print("\nCarpetas incompletas:", len(incompletas))
    display(incompletas.head(20))

auditoria.to_csv(
    SALIDA / "auditoria_teselas.csv",
    index=False
)

Carpetas encontradas: 644

Archivos encontrados:
completada    644
registro      644
csv           446
geojson       446
png           446
dtype: int64

Clastos conservados según registros:
47656

Carpetas incompletas: 198


,tesela,completada,registro,csv,geojson,png,granos,semillas
0,x00000_y00000,True,True,False,False,False,0,0
1,x00000_y00896,True,True,False,False,False,0,0
2,x00000_y01792,True,True,False,False,False,0,0
3,x00000_y02688,True,True,False,False,False,0,0
4,x00000_y03584,True,True,False,False,False,0,0
12,x00000_y10752,True,True,False,False,False,0,0
13,x00000_y11648,True,True,False,False,False,0,0
14,x00000_y12544,True,True,False,False,False,0,0
15,x00000_y13440,True,True,False,False,False,0,0
16,x00000_y14336,True,True,False,False,False,0,0


In [ ]:
import pandas as pd

archivos = sorted(
    SALIDA.glob("x*/mediciones.csv")
)

tablas = []

for archivo in archivos:
    try:
        df = pd.read_csv(archivo)

        if df.empty:
            continue

        df["archivo_origen"] = archivo.parent.name
        tablas.append(df)

    except Exception as e:
        print("No se pudo leer:", archivo, e)

if tablas:
    mediciones = pd.concat(
        tablas,
        ignore_index=True
    )

    if "id_grano" in mediciones.columns:
        mediciones = mediciones.drop_duplicates(
            subset="id_grano"
        )

    destino = SALIDA / "mediciones_consolidadas.csv"

    mediciones.to_csv(
        destino,
        index=False
    )

    print("Clastos:", len(mediciones))
    print("Columnas:", list(mediciones.columns))
    print("Archivo:", destino)

    display(mediciones.head())

else:
    print("No se encontraron mediciones con datos.")

No se pudo leer: /content/drive/MyDrive/gravas_mana/resultados_completo/x04480_y00000/mediciones.csv No columns to parse from file
No se pudo leer: /content/drive/MyDrive/gravas_mana/resultados_completo/x17024_y19712/mediciones.csv No columns to parse from file
No se pudo leer: /content/drive/MyDrive/gravas_mana/resultados_completo/x18816_y02688/mediciones.csv No columns to parse from file
Clastos: 47656
Columnas: ['area', 'centroid-0', 'centroid-1', 'perimeter', 'orientation', 'major_axis_length', 'minor_axis_length', 'max_intensity-0', 'min_intensity-0', 'mean_intensity-0', 'max_intensity-1', 'min_intensity-1', 'mean_intensity-1', 'max_intensity-2', 'min_intensity-2', 'mean_intensity-2', 'id_grano', 'tesela', 'pixel_col', 'pixel_fila', 'x_utm', 'y_utm', 'toca_borde', 'archivo_origen']
Archivo: /content/drive/MyDrive/gravas_mana/resultados_completo/mediciones_consolidadas.csv


,area,centroid-0,centroid-1,perimeter,orientation,major_axis_length,minor_axis_length,max_intensity-0,min_intensity-0,mean_intensity-0,...,min_intensity-2,mean_intensity-2,id_grano,tesela,pixel_col,pixel_fila,x_utm,y_utm,toca_borde,archivo_origen
0,0.000490,934.237853,915.584438,0.092081,-0.567135,0.035691,0.017822,214.0,67.0,137.863071,...,57.0,142.836100,x00000_y04480_g00001,x00000_y04480,915.584438,5414.237853,371855.643265,2.057661e+06,False,x00000_y04480
1,0.170624,589.544475,882.708922,3.378171,-0.288336,0.943246,0.273330,255.0,0.0,170.887234,...,0.0,147.002077,x00000_y04480_g00003,x00000_y04480,882.708922,5069.544475,371855.610390,2.057661e+06,True,x00000_y04480
2,0.009609,836.779871,943.313448,0.395636,-0.720491,0.135470,0.095087,218.0,13.0,125.994791,...,11.0,144.073654,x00000_y04480_g00005,x00000_y04480,943.313448,5316.779871,371855.670994,2.057661e+06,False,x00000_y04480
3,0.015261,923.488197,841.858645,0.501532,0.194807,0.170347,0.116008,252.0,26.0,162.213883,...,18.0,177.907820,x00000_y04480_g00006,x00000_y04480,841.858645,5403.488197,371855.569541,2.057661e+06,False,x00000_y04480
4,0.000757,165.775468,615.033993,0.150221,-0.257863,0.058754,0.019016,202.0,0.0,133.973333,...,0.0,105.186667,x00000_y05376_g00000,x00000_y05376,615.033993,5541.775468,371855.342722,2.057661e+06,False,x00000_y05376


In [ ]:
import json
from collections import Counter

estados = Counter()

for carpeta in carpetas:
    archivo = carpeta / "registro.json"

    if not archivo.exists():
        estados["sin_registro"] += 1
        continue

    datos = json.loads(archivo.read_text())

    estado = datos.get("estado", "sin_estado")
    estados[estado] += 1

print("Estados registrados:")
for estado, n in estados.items():
    print(f"{estado}: {n}")

Estados registrados:
sin_semillas: 2
sin_datos: 196
completada: 446


## 6. Consolidar en GeoPackage, CSV y tabla de control

Ejecutar después de completar las teselas deseadas. Se incluyen solo teselas con marcador `completado.ok`. La primera prueba con dos teselas produce un conjunto **parcial**, no la cartografía total.

In [ ]:
# Incluir TODAS las teselas con registro, incluso sin datos o sin semillas.
archivos_registro = [SALIDA/v['id']/'registro.json' for v in plan
                     if (SALIDA/v['id']/'completado.ok').exists()
                     and (SALIDA/v['id']/'registro.json').exists()]
registros = [json.loads(p.read_text()) for p in archivos_registro]
pd.DataFrame(registros).to_csv(SALIDA/'control_teselas.csv', index=False)
print('Teselas registradas:', len(registros), 'de', len(plan))

# Solo las teselas con geometrías exportadas se consolidan.
rutas = [SALIDA/v['id']/'granos_utm.geojson' for v in plan
         if (SALIDA/v['id']/'completado.ok').exists()
         and (SALIDA/v['id']/'granos_utm.geojson').exists()]
partes = []
for ruta in rutas:
    gdf_parte = gpd.read_file(ruta)
    if not gdf_parte.empty:
        partes.append(gdf_parte)
if partes:
    unidos = gpd.GeoDataFrame(pd.concat(partes, ignore_index=True), crs=EPSG)
    unidos.to_file(SALIDA/'granos_mana.gpkg', layer='granos', driver='GPKG')
    unidos.drop(columns='geometry').to_csv(SALIDA/'granos_mana.csv', index=False)
    print('Clastos exportados:', len(unidos))
    print('GPKG:', SALIDA/'granos_mana.gpkg')
else:
    print('Todavía no hay clastos exportables. Revisar control_teselas.csv')


Teselas registradas: 644 de 644
Clastos exportados: 47656
GPKG: /content/drive/MyDrive/gravas_mana/resultados_completo/granos_mana.gpkg


## 7. Comprobaciones y ejecución completa

1. **Piloto:** mantener `MODO_PRUEBA=True` y `MAX_TESELAS=2`. Se priorizan dos teselas próximas a la región que ya produjo resultados.
2. Revisar `segmentacion.png`, `granos_utm.geojson`, `mediciones.csv` y `registro.json` de una tesela con clastos.
3. **Procesamiento completo:** cambiar `MODO_PRUEBA=False` y `MAX_TESELAS=None`, volver a ejecutar las celdas de configuración, planificación y procesamiento. Con `REANUDAR=True`, las teselas ya registradas se omiten.
4. Ejecutar la consolidación periódicamente. `control_teselas.csv` incluye teselas `sin_datos` y `sin_semillas`, que no producen geometrías.
5. El descarte de zonas negras se basa en un umbral configurable, no en una máscara oficial de nodata; inspeccionar muestras descartadas. `sin_semillas` no significa ausencia demostrada de clastos.
6. La asignación por centroide evita duplicar los mismos objetos entre teselas, pero **no reconstruye clastos cortados** ni garantiza una correspondencia perfecta entre máscaras vecinas.

**Nota:** El JPEG grande puede necesitar decodificación repetida. El cuaderno no se ha ejecutado aquí con el ortomosaico completo ni con los modelos en GPU.
